# 📱 スマホだけでセットアップ（Garmin週次レポート）

このノートブックを上から順に実行すると、**PCなし・スマホだけ**で次が完了します。

1. Garmin の認証トークンを生成
2. GitHub にリポジトリを作成し、必要ファイルを自動アップロード
3. （最後だけ手動）GitHub に秘密情報を登録 → 週次自動実行を開始

---
### 事前に用意するもの（すべてスマホのブラウザで取得できます）
- **Google アカウント**（このColabを動かすため）
- **Garmin Connect** のメール・パスワード
- **Gemini APIキー** … `aistudio.google.com` →「Get API key」
- **LINE** … `developers.line.biz` でチャネルアクセストークンと自分のユーザーID
- **GitHub アカウント** と **個人アクセストークン(PAT)**
  - `github.com` → Settings → Developer settings → Personal access tokens →
    **Tokens (classic)** → Generate new token → スコープ **repo** と **workflow** にチェック

> 各キーの詳しい取得手順は、アップロードされる `README.md` の第2章にもあります。

▶ セル左の実行ボタンを上から順に押してください。


## 手順1: 必要ライブラリのインストール

In [ ]:
!pip install garminconnect requests -q
print('✅ インストール完了')

## 手順2: Garmin トークンを生成

Garmin のメール・パスワード（2段階認証ありの場合はコードも）を入力します。
成功すると長いトークン文字列が `GARMIN_TOKENS` 変数に入ります（手順3で自動利用）。

> 補足: ColabはGoogleのデータセンターから動くため、ごく稀にGarminのログインが
> 弾かれることがあります。その場合は時間をおいて再実行してください。


In [ ]:
import getpass
from garminconnect import Garmin

email = input("Garmin email: ").strip()
password = getpass.getpass("Garmin password: ")
g = Garmin(email, password, prompt_mfa=lambda: input("MFAコード（求められたら入力）: ").strip())
g.login()
GARMIN_TOKENS = g.client.dumps()
print("✅ Garminログイン成功。トークンを取得しました（長さ:", len(GARMIN_TOKENS), "文字）")


## 手順3: GitHub リポジトリを作成し、ファイルを自動アップロード

GitHub の **個人アクセストークン(PAT)** と、作成したい **リポジトリ名** を入力します。
リポジトリ（private）が作られ、必要なファイル一式が自動で配置されます。


In [ ]:
PROJECT_FILES_B64 = {
    'garmin_weekly_report.py': '...',
    'requirements.txt': 'IyBHYXJtaW4gQ29ubmVjdCDpgLHmrKHjg6njg7Pjg4vjg7PjgrDjg6zjg53jg7zjg4gg5L6d5a2Y44OR44OD44Kx44O844K4CiMgICDjgqTjg7Pjgrnjg4jjg7zjg6s6IHBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy50eHQKCiMgR2FybWluIENvbm5lY3Qg6Z2e5YWs5byPQVBJ44Op44OD44OR44O877yIMC4zLjDku6XpmY3jga/jg43jgqTjg4bjgqPjg5boqo3oqLzvvIkKZ2FybWluY29ubmVjdD49MC4zLjAKCiMgSFRUUOODquOCr+OCqOOCueODiO+8iEdlbWluaSBBUEkgLyBMSU5FIEFQSSDlkbzjgbPlh7rjgZfnlKjvvIkKcmVxdWVzdHM+PTIuMzEuMAoKIyDku6XkuIvjga8gZ2FybWluY29ubmVjdCDjgYzoh6rli5XjgaflhaXjgozjgovkvp3lrZjvvIjmmI7npLrjgZfjgabjgYrjgY/jgajnkrDlooPlt67nlbDjgavlvLfjgYTvvIkKY3VybF9jZmZpCnVhLWdlbmVyYXRvcgo=',
    'config.example.ini': 'IyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyAg6Kit5a6a44OV44Kh44Kk44Or77yI44GT44Gu44OV44Kh44Kk44Or44KSIGNvbmZpZy5pbmkg44Go44GE44GG5ZCN5YmN44Gn44Kz44OU44O844GX44Gm5L2/44GE44G+44GZ77yJCiMKIyAgIGNwIGNvbmZpZy5leGFtcGxlLmluaSBjb25maWcuaW5pICAg77yITWFjIC8gTGludXjvvIkKIyAgIGNvcHkgY29uZmlnLmV4YW1wbGUuaW5pIGNvbmZpZy5pbmkg77yIV2luZG93c++8iQojCiMgIOWQhOWApOOBruWPluW+l+aWueazleOBryBSRUFETUUubWQg44KS5Y+C54Wn44GX44Gm44GP44Gg44GV44GE44CCCiMgIGNvbmZpZy5pbmkg44Gr44Gv44OR44K544Ov44O844OJ44O7QVBJ44Kt44O844GM5ZCr44G+44KM44G+44GZ44CC57W25a++44Gr5LuW5Lq644Gr5rih44GX44Gf44KKCiMgIEdpdEh1YuetieOBq+WFrOmWi+OBl+OBn+OCiuOBl+OBquOBhOOBp+OBj+OBoOOBleOBhO+8iC5naXRpZ25vcmUg44Gn6Zmk5aSW5riI44G/77yJ44CCCiMKIyAg4oC7IEdpdEh1YiBBY3Rpb25z562J44Gu44Kv44Op44Km44OJ5a6f6KGM44Gn44GvIGNvbmZpZy5pbmkg44Gv5LiN6KaB44Gn44GZ44CCCiMgICAg5ZCM44GY6aCF55uu44KS55Kw5aKD5aSJ5pWw77yIU2VjcmV0c++8ieOBp+a4oeOBm+OBvuOBme+8iOeSsOWig+WkieaVsOOBjOWEquWFiOOBleOCjOOBvuOBme+8ieOAggojICAgIOWvvuW/nOOBmeOCi+eSsOWig+WkieaVsOWQjeOBryBSRUFETUUubWQg44Gu44CMUEPjg6zjgrnpgYvnlKjjgI3jgpLlj4LnhafjgIIKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKCltnYXJtaW5dCiMgR2FybWluIENvbm5lY3Qg44Gu44Ot44Kw44Kk44Oz5oOF5aCxCmVtYWlsICAgID0geW91cl9lbWFpbEBleGFtcGxlLmNvbQpwYXNzd29yZCA9IHlvdXJfcGFzc3dvcmQKCltnZW1pbmldCiMgR29vZ2xlIEFJIFN0dWRpbyDjgaflj5blvpfjgZfjgZ9BUEnjgq3jg7zvvIjjgq/jg6zjgqvkuI3opoHjg7vnhKHmlpnmnqDvvIkKYXBpX2tleSA9IHlvdXJfZ2VtaW5pX2FwaV9rZXkKIyDkvb/nlKjjg6Ljg4fjg6vvvIjnhKHmlpnmnqA6IGdlbWluaS0yLjUtZmxhc2ggLyDjgZXjgonjgavou73ph486IGdlbWluaS0yLjUtZmxhc2gtbGl0Ze+8iQptb2RlbCAgID0gZ2VtaW5pLTIuNS1mbGFzaAoKW2xpbmVdCiMgTElORSBNZXNzYWdpbmcgQVBJIOOBruODgeODo+ODjeODq+OCouOCr+OCu+OCueODiOODvOOCr+ODs++8iOmVt+acn++8ieOBqOOAgeiHquWIhuOBruODpuODvOOCtuODvElECmNoYW5uZWxfYWNjZXNzX3Rva2VuID0geW91cl9saW5lX2NoYW5uZWxfYWNjZXNzX3Rva2VuCnVzZXJfaWQgICAgICAgICAgICAgID0geW91cl9saW5lX3VzZXJfaWQKCltnb2FsXQojIOebruaomeOBqOODl+ODreODleOCo+ODvOODq++8iOiHqueUseOBq+abuOOBjeaPm+OBiOOBpuOBj+OBoOOBleOBhOOAgkxMTeOBruWIhuaekOOBq+a4oeOBleOCjOOBvuOBme+8iQptYXJhdGhvbl90aW1lID0gM+aZgumWkzMw5YiGCnJhY2VfcGFjZSAgICAgPSA0OjU4L2ttCiMg6KSH5pWw6KGM44Gn5pu444GP5aC05ZCI44Gv44CBMuihjOebruS7pemZjeOCkuW/heOBmuWNiuinkuOCueODmuODvOOCueOBp+Wtl+S4i+OBkuOBl+OBpuOBj+OBoOOBleOBhApwcm9maWxlID0KICAgIC0g55uu5qiZOiDjg5Xjg6vjg57jg6njgr3jg7Mg44K144OWMy4177yIM+aZgumWkzMw5YiG5YiH44KK77yJCiAgICAtIOW/heimgeODrOODvOOCueODmuODvOOCuTogNDo1OC9rbQogICAgLSDjg4jjg6zjg7zjg4vjg7PjgrDmlrnph506IDgwLzIw44Or44O844Or77yI5pyJ6YW457SgODAl44CB6auY5by35bqmMjAl77yJCiAgICAtIOODreODs+OCsOi1sOebruaomTog6YCxMeWbniAyMOOAnDMwa20KICAgIC0g6YCx6ZaT6LWw6KGM6Led6Zui55uu5qiZOiA1MOOAnDcwa20KICAgIC0g44Os44O844K55pelOiAyMDI25bm0MTHmnIgz5pel44CA576k6aas44Oe44Op44K944OzCg==',
    'README.md': '...',
    '.gitignore': 'IyDoqo3oqLzmg4XloLHvvIjntbblr77jgavjgrPjg5/jg4Pjg4jjgZfjgarjgYTvvIkKY29uZmlnLmluaQoKIyDlrp/ooYzjg63jgrAKZ2FybWluX3JlcG9ydC5sb2cKCiMgR2FybWluIOiqjeiovOODiOODvOOCr+ODs+OBruOCreODo+ODg+OCt+ODpQouZ2FybWluY29ubmVjdC8KZ2FybWluX3Rva2Vucy5qc29uCgojIFB5dGhvbgpfX3B5Y2FjaGVfXy8KKi5weWMKLnZlbnYvCnZlbnYvCmVudi8KCiMgT1MKLkRTX1N0b3JlClRodW1icy5kYgo=',
    'setup_garmin_token.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKR2FybWluIOODiOODvOOCr+ODs+eUn+aIkOODhOODvOODq++8iOacgOWIneOBqzHlm57jgaDjgZHjgIHoh6rliIbjga5QQ+OBp+Wun+ihjO+8iQoKR2l0SHViIEFjdGlvbnMg44Gq44Gp44Gu44Kv44Op44Km44OJ44Gn5q+O5Zue44OR44K544Ov44O844OJ44Ot44Kw44Kk44Oz44GZ44KL44Go44CBCkdhcm1pbiDjgavjg5zjg4Pjg4jmibHjgYTjgZXjgozjgabjg5bjg63jg4Pjgq/jgZXjgozjgoTjgZnjgY/jgarjgorjgb7jgZnjgIIK44GT44Gu44K544Kv44Oq44OX44OI44Gn5LiA5bqm44Gg44GR44Ot44Kw44Kk44Oz44GX44Gm44OI44O844Kv44Oz5paH5a2X5YiX44KS55Sf5oiQ44GX44CBCuOBneOCjOOCkiBHaXRIdWIgU2VjcmV0IGBHQVJNSU5fVE9LRU5TYCDjgavnmbvpjLLjgZnjgovjgajjgIEK44Kv44Op44Km44OJ5YG044Gv44OR44K544Ov44O844OJ54Sh44GX44O744OI44O844Kv44Oz44Gg44GR44Gn44Ot44Kw44Kk44Oz44Gn44GN44G+44GZ44CCCgrkvb/jgYTmlrk6CiAgICBweXRob24gc2V0dXBfZ2FybWluX3Rva2VuLnB5CiAgICDvvIjjg6Hjg7zjg6vjg7vjg5Hjgrnjg6/jg7zjg4njgIHlv4XopoHjgarjgolNRkHjgrPjg7zjg4njgpLlhaXlipvvvIkKICAgIOKGkiDooajnpLrjgZXjgozjgZ/plbfjgYTmloflrZfliJfjgpLjgZnjgbnjgabjgrPjg5Tjg7zjgZfjgaYgU2VjcmV0IGBHQVJNSU5fVE9LRU5TYCDjgavosrzjgorku5jjgZEKCuODiOODvOOCr+ODs+OBruacieWKueacn+mZkOOBr+e0hDHlubTjgafjgZnjgILmnJ/pmZDliIfjgozjgoTjg63jgrDjgqTjg7PlpLHmlZfjgYzlh7rjgZ/jgonlho3lrp/ooYzjgZfjgabjgY/jgaDjgZXjgYTjgIIKIiIiCgppbXBvcnQgZ2V0cGFzcwppbXBvcnQgc3lzCgoKZGVmIG1haW4oKToKICAgIHRyeToKICAgICAgICBmcm9tIGdhcm1pbmNvbm5lY3QgaW1wb3J0IEdhcm1pbgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHN5cy5leGl0KCJnYXJtaW5jb25uZWN0IOOBjOacquOCpOODs+OCueODiOODvOODq+OBp+OBmeOAguWFiOOBqyBgcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLnR4dGAg44KS5a6f6KGM44GX44Gm44GP44Gg44GV44GE44CCIikKCiAgICBwcmludCgiPT09IEdhcm1pbiDjg4jjg7zjgq/jg7PnlJ/miJAgPT09IikKICAgIGVtYWlsID0gaW5wdXQoIkdhcm1pbiBlbWFpbDogIikuc3RyaXAoKQogICAgcGFzc3dvcmQgPSBnZXRwYXNzLmdldHBhc3MoIkdhcm1pbiBwYXNzd29yZDogIikKCiAgICAjIE1GQe+8iDLmrrXpmo7oqo3oqLzvvInjgYzmnInlirnjgarloLTlkIjjga/jgrPjg7zjg4nlhaXlipvjgpLkv4PjgZkKICAgIGdhcm1pbiA9IEdhcm1pbihlbWFpbCwgcGFzc3dvcmQsIHByb21wdF9tZmE9bGFtYmRhOiBpbnB1dCgiTUZB44Kz44O844OJ44KS5YWl5YqbOiAiKS5zdHJpcCgpKQoKICAgIHByaW50KCJcbuODreOCsOOCpOODs+S4rS4uLiIpCiAgICB0cnk6CiAgICAgICAgZ2FybWluLmxvZ2luKCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBzeXMuZXhpdChmIuKdjCDjg63jgrDjgqTjg7PjgavlpLHmlZfjgZfjgb7jgZfjgZ86IHtlfSIpCgogICAgdG9rZW5fc3RyID0gZ2FybWluLmNsaWVudC5kdW1wcygpCgogICAgcHJpbnQoIlxu4pyFIOODreOCsOOCpOODs+aIkOWKn+OAguS4i+OBrjHooYzvvIjplbfjgYTmloflrZfliJfvvInjgpLjgZnjgbnjgabjgrPjg5Tjg7zjgZfjgabjgIEiKQogICAgcHJpbnQoIiAgIEdpdEh1YiBTZWNyZXTjgI5HQVJNSU5fVE9LRU5T44CP44Gr6LK844KK5LuY44GR44Gm44GP44Gg44GV44GE44CCIikKICAgIHByaW50KCIgICDvvIjjg63jg7zjgqvjg6vjgafjg4jjg7zjgq/jg7PpgYvnlKjjgZfjgZ/jgYTloLTlkIjjga/nkrDlooPlpInmlbAgR0FSTUlOX1RPS0VOUyDjgavoqK3lrprvvIlcbiIpCiAgICBwcmludCgi4pSAIiAqIDYwKQogICAgcHJpbnQodG9rZW5fc3RyKQogICAgcHJpbnQoIuKUgCIgKiA2MCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    '.github/workflows/weekly-report.yml': 'bmFtZTogV2Vla2x5IFJ1bm5pbmcgUmVwb3J0CgojIOWun+ihjOOCv+OCpOODn+ODs+OCsApvbjoKICBzY2hlZHVsZToKICAgICMgR2l0SHViIEFjdGlvbnMg44GuIGNyb24g44GvIFVUQyDln7rmupbjgafjgZnjgIIKICAgICMg5pel5pys5pmC6ZaT77yISlNUPVVUQys577yJ5pyI5pucIDA3OjAwIOKGkiBVVEMg5pel5pucIDIyOjAwCiAgICAtIGNyb246ICIwIDIyICogKiAwIgogIHdvcmtmbG93X2Rpc3BhdGNoOiAgICMg5omL5YuV5a6f6KGM44Oc44K/44Oz77yIQWN0aW9uc+OCv+ODluOBi+OCieS7iuOBmeOBkOWun+ihjOOBp+OBjeOCi++8j+WLleS9nOeiuuiqjeeUqO+8iQoKam9iczoKICByZXBvcnQ6CiAgICBydW5zLW9uOiB1YnVudHUtbGF0ZXN0CiAgICBzdGVwczoKICAgICAgLSBuYW1lOiDjg6rjg53jgrjjg4jjg6rjgpLlj5blvpcKICAgICAgICB1c2VzOiBhY3Rpb25zL2NoZWNrb3V0QHY0CgogICAgICAtIG5hbWU6IFB5dGhvbiDjgpLjgrvjg4Pjg4jjgqLjg4Pjg5cKICAgICAgICB1c2VzOiBhY3Rpb25zL3NldHVwLXB5dGhvbkB2NQogICAgICAgIHdpdGg6CiAgICAgICAgICBweXRob24tdmVyc2lvbjogIjMuMTIiCiAgICAgICAgICBjYWNoZTogInBpcCIKCiAgICAgIC0gbmFtZTog5L6d5a2Y44OR44OD44Kx44O844K444KS44Kk44Oz44K544OI44O844OrCiAgICAgICAgcnVuOiBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMudHh0CgogICAgICAtIG5hbWU6IOODrOODneODvOODiOWun+ihjAogICAgICAgIGVudjoKICAgICAgICAgICMg4pSA4pSAIFNlY3JldHPvvIhTZXR0aW5ncyA+IFNlY3JldHMgYW5kIHZhcmlhYmxlcyA+IEFjdGlvbnMgPiBTZWNyZXRz77yJ4pSA4pSACiAgICAgICAgICBHQVJNSU5fVE9LRU5TOiAgICAgICAgICAgICAke3sgc2VjcmV0cy5HQVJNSU5fVE9LRU5TIH19CiAgICAgICAgICBHQVJNSU5fRU1BSUw6ICAgICAgICAgICAgICAke3sgc2VjcmV0cy5HQVJNSU5fRU1BSUwgfX0KICAgICAgICAgIEdBUk1JTl9QQVNTV09SRDogICAgICAgICAgICR7eyBzZWNyZXRzLkdBUk1JTl9QQVNTV09SRCB9fQogICAgICAgICAgR0VNSU5JX0FQSV9LRVk6ICAgICAgICAgICAgJHt7IHNlY3JldHMuR0VNSU5JX0FQSV9LRVkgfX0KICAgICAgICAgIExJTkVfQ0hBTk5FTF9BQ0NFU1NfVE9LRU46ICR7eyBzZWNyZXRzLkxJTkVfQ0hBTk5FTF9BQ0NFU1NfVE9LRU4gfX0KICAgICAgICAgIExJTkVfVVNFUl9JRDogICAgICAgICAgICAgICR7eyBzZWNyZXRzLkxJTkVfVVNFUl9JRCB9fQogICAgICAgICAgIyDilIDilIAgVmFyaWFibGVz77yI5Lu75oSP44CC5ZCM55S76Z2i44GuIFZhcmlhYmxlcyDjgr/jg5bjgILnp5jlr4bjgafjgarjgYToqK3lrprvvInilIDilIAKICAgICAgICAgIEdFTUlOSV9NT0RFTDogICAgICAgICAgICAgICR7eyB2YXJzLkdFTUlOSV9NT0RFTCB9fQogICAgICAgICAgR09BTF9NQVJBVEhPTl9USU1FOiAgICAgICAgJHt7IHZhcnMuR09BTF9NQVJBVEhPTl9USU1FIH19CiAgICAgICAgICBHT0FMX1JBQ0VfUEFDRTogICAgICAgICAgICAke3sgdmFycy5HT0FMX1JBQ0VfUEFDRSB9fQogICAgICAgICAgUlVOTkVSX1BST0ZJTEU6ICAgICAgICAgICAgJHt7IHZhcnMuUlVOTkVSX1BST0ZJTEUgfX0KICAgICAgICBydW46IHB5dGhvbiBnYXJtaW5fd2Vla2x5X3JlcG9ydC5weQo=',
}

import base64, getpass, json, requests

PAT = getpass.getpass("GitHub Personal Access Token (classic, repo+workflow): ").strip()
REPO = input("作成するリポジトリ名（例: garmin-weekly-report）: ").strip() or "garmin-weekly-report"

H = {"Authorization": f"Bearer {PAT}",
     "Accept": "application/vnd.github+json",
     "X-GitHub-Api-Version": "2022-11-28"}

# ユーザー名を取得
me = requests.get("https://api.github.com/user", headers=H)
me.raise_for_status()
owner = me.json()["login"]
print("GitHubユーザー:", owner)

# リポジトリ作成（既存ならスキップ）
r = requests.post("https://api.github.com/user/repos", headers=H,
                  json={"name": REPO, "private": True,
                        "description": "Garmin weekly running report (Gemini + LINE)"})
if r.status_code == 201:
    print("✅ リポジトリ作成:", r.json()["html_url"])
elif r.status_code == 422:
    print("ℹ️ 同名リポジトリが既にあります。そこに上書きします。")
else:
    r.raise_for_status()

# ファイルを1つずつ Contents API でアップロード
def put_file(path, content_b64):
    url = f"https://api.github.com/repos/{owner}/{REPO}/contents/{path}"
    # 既存ファイルがあれば sha を取得して更新
    sha = None
    ex = requests.get(url, headers=H)
    if ex.status_code == 200:
        sha = ex.json().get("sha")
    body = {"message": f"add {path}", "content": content_b64, "branch": "main"}
    if sha:
        body["sha"] = sha
    resp = requests.put(url, headers=H, json=body)
    resp.raise_for_status()

for path, b64 in PROJECT_FILES_B64.items():
    put_file(path, b64)
    print("  ⬆️", path)

print("\n✅ 全ファイルのアップロード完了")
print("リポジトリ:", f"https://github.com/{owner}/{REPO}")


## 手順4: 秘密情報を登録して自動実行を開始（スマホのブラウザで）

ファイルは配置できました。最後に GitHub 側で秘密情報を登録します。
（安全のため、ここだけは手動で行います）

1. 作成されたリポジトリを開く（手順3の最後に出たURL）
2. **Settings → Secrets and variables → Actions** を開く
3. **Secrets** タブで「New repository secret」から次の4つを登録:

   | Name | Value |
   |------|-------|
   | `GARMIN_TOKENS` | 手順2で生成したトークン文字列（下のセルを実行すると再表示できます） |
   | `GEMINI_API_KEY` | Gemini APIキー |
   | `LINE_CHANNEL_ACCESS_TOKEN` | LINEチャネルアクセストークン |
   | `LINE_USER_ID` | 自分のLINEユーザーID（U始まり） |

4. （任意）**Variables** タブで目標を登録: `GOAL_MARATHON_TIME`, `GOAL_RACE_PACE`, `RUNNER_PROFILE`, `GEMINI_MODEL`
5. **Actions** タブを開き、ワークフローを有効化
6. 「Weekly Running Report」→「**Run workflow**」で手動実行 → LINEに届けば成功 🎉

以降は毎週月曜 7:00（日本時間）に自動でレポートが届きます。


In [ ]:
# GARMIN_TOKENS をもう一度表示（コピーして Secret に貼り付け）
print(GARMIN_TOKENS)
